# 00. 공통 부품 설명 — `config.py`와 `dart_api.py`

**이 노트북이 하는 일:** 01~04번 노트북이 모두 가져다 쓰는 두 개의 `.py` 파일이 무엇을 하는지, 한 줄씩 풀어서 설명합니다.

| 항목 | 내용 |
|---|---|
| 입력 | `config.py`, `dart_api.py` (같은 폴더) |
| 출력 | 없음 (화면에 설명만 나옴) |
| DART API 호출 | **하지 않습니다.** 가짜 응답으로 시험합니다. 인증키도 필요 없습니다 |
| 실행 시간 | 1~2초 |
| 먼저 알아야 할 것 | 없음 (이 노트북이 처음부터 설명합니다) |

## 왜 이 둘만 노트북이 아니라 `.py` 파일인가?
01~04번 노트북이 **똑같은 설정값과 똑같은 "요청 보내는 도구"** 를 써야 합니다. 노트북마다 복사해 두면 한쪽만 고쳐서 서로 달라질 수 있습니다. 그래서 한 곳(`.py`)에 두고 `import`로 불러 씁니다.

비유: `config.py`는 **레시피 카드의 재료 목록**, `dart_api.py`는 **주방 도구(칼·냄비)** 입니다. 요리(01~04번)는 노트북마다 다르지만 재료 목록과 도구는 공유합니다.

## 이 노트북의 지도
1. `config.py` — 설정값 하나하나의 뜻
2. 계정명 규칙(`ACCOUNT_PATTERNS`) — 정규식이 뭔지, 왜 공백을 지우는지
3. `dart_api.py` — 요청을 보내고, 응답의 상태코드를 처리하는 방법 (가짜 응답으로 시연)
4. 어떤 노트북이 어떤 부품을 쓰는지

In [1]:
import os, sys, re, json
from pathlib import Path
from unittest import mock

sys.path.insert(0, str(Path.cwd()))   # 같은 폴더의 config.py, dart_api.py를 불러오기 위해
import config
import dart_api
print("불러오기 성공")

불러오기 성공


---
# 1. `config.py` — 설정값 모음

설정값을 한 곳에 모아 두면 "이 숫자가 어디서 왔지?"를 찾기 쉽습니다. 값을 바꾸면 사전 고정 문서의 변경 기록에도 적는다는 원칙이 파일 맨 위 설명에 있습니다.

In [2]:
import pandas as pd

rows = [
    ("RAW",   "Corp/" + config.RAW.relative_to(config.BASE).as_posix(),   "DART에서 받은 원본 응답을 두는 곳. **수정하지 않는다**"),
    ("OUT",   "Corp/" + config.OUT.relative_to(config.BASE).as_posix(),   "우리가 만든 결과표(CSV)를 두는 곳"),
    ("YEARS", config.YEARS, "수집할 사업연도. 후행 4분기 합(TTM)을 만들려면 분석 시작 전 해가 필요해서 2018부터"),
    ("REPORTS", config.REPORTS, "보고서 이름 → DART 보고서 코드. Q1=1분기, H1=반기, Q3=3분기, FY=사업(연간). 4분기 보고서는 따로 없음"),
    ("FS_ORDER", config.FS_ORDER, "연결(CFS)을 먼저 묻고, 없으면 별도(OFS)를 묻는 순서"),
    ("SLEEP", config.SLEEP, "호출 사이에 쉬는 초. 서버에 무리를 주지 않으려는 값 (임의)"),
    ("N_COMPANY_PROBE", config.N_COMPANY_PROBE, "업종코드 체계를 눈으로 보려고 무작위로 뽑아 본 기업 수 (임의)"),
    ("N_PILOT", config.N_PILOT, "파일럿 표본 기업 수 (임의, 통계적 근거 없음)"),
    ("SEED", config.SEED, "무작위 추출 재현용 번호. 같은 번호를 쓰면 같은 기업이 뽑힘"),
]
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(rows, columns=["이름", "현재 값", "뜻"])

,이름,현재 값,뜻
0,RAW,Corp/data/raw/DART_파일럿,DART에서 받은 원본 응답을 두는 곳. **수정하지 않는다**
1,OUT,Corp/outputs/01_파일럿,우리가 만든 결과표(CSV)를 두는 곳
2,YEARS,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]",수집할 사업연도. 후행 4분기 합(TTM)을 만들려면 분석 시작 전 해가 필요해서 2018부터
3,REPORTS,"{'Q1': '11013', 'H1': '11012', 'Q3': '11014', 'FY': '11011'}","보고서 이름 → DART 보고서 코드. Q1=1분기, H1=반기, Q3=3분기, FY=사업(연간). 4분기 보고서는 따로 없음"
4,FS_ORDER,"[CFS, OFS]","연결(CFS)을 먼저 묻고, 없으면 별도(OFS)를 묻는 순서"
5,SLEEP,0.4,호출 사이에 쉬는 초. 서버에 무리를 주지 않으려는 값 (임의)
6,N_COMPANY_PROBE,150,업종코드 체계를 눈으로 보려고 무작위로 뽑아 본 기업 수 (임의)
7,N_PILOT,40,"파일럿 표본 기업 수 (임의, 통계적 근거 없음)"
8,SEED,20261102,무작위 추출 재현용 번호. 같은 번호를 쓰면 같은 기업이 뽑힘


### 해석
- **경로(`RAW`, `OUT`)** 는 `Corp/data/raw/DART_파일럿`, `Corp/outputs/01_파일럿`입니다. 원래 `pilot/` 폴더 안에 있던 `raw/`, `out/`을 Corp 폴더 정리 때 옮기면서 이 경로 한 줄만 바꿨습니다.
- **`YEARS`가 2018부터인 이유:** 분석 창은 2019Q1부터인데, "후행 4분기 합"을 2019Q1에 만들려면 2018Q2~Q4가 필요해서입니다.
- **`SLEEP`, `N_COMPANY_PROBE`, `N_PILOT`은 임의로 정한 값입니다.** 공식 기준도 통계적 근거도 아닙니다. 다른 값을 써도 결론이 달라지는 종류의 숫자가 아니라 "작업 규모"를 정하는 숫자입니다.
- **`SEED`** 덕분에 02번 노트북에서 "파일럿 40곳이 어떻게 뽑혔는지"를 똑같이 재현해 볼 수 있습니다.

In [3]:
print("제조업 코드 접두어 (induty_code 앞 2자리):", config.MANUFACTURING_PREFIXES[0], "~", config.MANUFACTURING_PREFIXES[-1], f"({len(config.MANUFACTURING_PREFIXES)}개)")
print()
print("이름으로 스팩/지주회사를 표시하는 규칙:")
for k, v in config.FLAG_NAME_PATTERNS.items():
    print(f"  {k:<8} {v}")

제조업 코드 접두어 (induty_code 앞 2자리): 10 ~ 34 (25개)

이름으로 스팩/지주회사를 표시하는 규칙:
  spac     스팩|SPAC|제\s*\d+\s*호
  holding  홀딩스|지주|Holdings


### 해석
- 제조업을 "업종코드 앞 2자리가 10~34"로 정한 것은 **150곳을 관찰해서 잠정으로 정한 값**입니다(사전 고정 문서 A1, 팀 확인 필요). DART 가이드에 업종코드 체계 설명이 없어서 실제 값을 보고 추정했습니다.
- 스팩·지주회사는 **이름만 보고 표시**합니다. 자동으로 빼지는 않고, 최종 판단은 사람이 합니다. 이름 규칙은 완벽하지 않습니다(예: 이름에 '지주'가 들어 있어도 지주회사가 아닐 수 있음).

---
# 2. 계정명 규칙 `ACCOUNT_PATTERNS` — 이 연구의 "번역 사전"

DART 응답에는 한 보고서에 계정이 100줄 넘게 들어 있습니다. 우리는 그중 **9개만** 필요합니다. 그런데 **계정 이름이 기업마다 조금씩 다릅니다**(예: '영업이익' / '영업이익(손실)', '금융비용' / '금융원가').

그래서 "이런 모양의 이름이면 이 계정으로 본다"는 규칙을 **정규식**으로 만들어 두었습니다.

### 정규식이란? (1분 설명)
글자 모양을 설명하는 짧은 암호입니다.
- `^영업이익` : **영업이익으로 시작**하는 글자 → '영업이익', '영업이익(손실)' 모두 해당
- `^자산총계$` : **자산총계 그것만** (앞뒤에 다른 글자 없음)
- `(금융비용|금융원가)` : 금융비용 **또는** 금융원가
- `이자.{0,2}지급` : '이자' 다음에 아무 글자 0~2개 뒤 '지급' → '이자지급', '이자의 지급', '이자지급(영업)'

In [4]:
rows = []
for name, (sj, pat) in config.ACCOUNT_PATTERNS.items():
    rows.append((name, sj, pat))
pd.DataFrame(rows, columns=["우리가 부르는 이름", "어느 표? (sj_div)", "계정명 규칙(정규식)"])

,우리가 부르는 이름,어느 표? (sj_div),계정명 규칙(정규식)
0,assets,BS,^자산총계$
1,revenue,IS|CIS,^(매출액|수익\(매출액\)|영업수익)
2,op_income,IS|CIS,^영업이익
3,interest_exp,IS|CIS,^이자비용
4,fin_cost,IS|CIS,^(금융비용|금융원가)
5,interest_paid_cf,CF,"이자.{0,2}지급"
6,st_borrow,BS,^단기차입금
7,cur_ltd,BS,^(유동성장기차입금|유동성장기부채|유동성차입금)
8,cur_bond,BS,^(유동성사채|유동성장기사채)


표 이름(`sj_div`)의 뜻: **BS** 재무상태표, **IS/CIS** 손익계산서(포괄손익계산서), **CF** 현금흐름표. `IS|CIS`는 "IS 또는 CIS"입니다. 같은 이름의 계정이 다른 표에 또 나오는 것을 막으려고 표 구분도 함께 겁니다(예: 현금흐름표에도 '이자비용'이라는 이름이 있음).

### 실제로 규칙이 어떻게 동작하는지 눈으로 확인

In [5]:
def norm(name):
    # 계정명의 공백을 모두 지운다. 실제 응답에 '유동성 장기부채'처럼 띄어쓰기가 들어간 이름이 있었다.
    return re.sub(r"\s+", "", name)

tests = [
    ("IS",  "영업이익(손실)",   "op_income"),
    ("IS",  "영업이익",         "op_income"),
    ("IS",  "금융원가",         "fin_cost"),
    ("CIS", "금융비용",         "fin_cost"),
    ("CF",  "이자지급(영업)",   "interest_paid_cf"),
    ("CF",  "이자비용",         "interest_exp"),     # 현금흐름표의 '이자비용' → 손익계산서 규칙에 안 걸려야 함
    ("BS",  "유동성 장기부채",  "cur_ltd"),          # 띄어쓰기 있는 이름
    ("BS",  "유동 차입금",      "cur_ltd"),          # 규칙에 없는 변형
    ("BS",  "단기차입금",       "st_borrow"),
]
out = []
for sj, nm, target in tests:
    want_sj, pat = config.ACCOUNT_PATTERNS[target]
    ok = bool(re.fullmatch(want_sj, sj)) and bool(re.search(pat, norm(nm)))
    out.append((sj, nm, target, "잡힘" if ok else "못 잡음"))
pd.DataFrame(out, columns=["표", "응답의 계정명", "시험할 대상", "결과"])

,표,응답의 계정명,시험할 대상,결과
0,IS,영업이익(손실),op_income,잡힘
1,IS,영업이익,op_income,잡힘
2,IS,금융원가,fin_cost,잡힘
3,CIS,금융비용,fin_cost,잡힘
4,CF,이자지급(영업),interest_paid_cf,잡힘
5,CF,이자비용,interest_exp,못 잡음
6,BS,유동성 장기부채,cur_ltd,잡힘
7,BS,유동 차입금,cur_ltd,못 잡음
8,BS,단기차입금,st_borrow,잡힘


### 해석
- `'영업이익(손실)'`이 잡히는 이유: 규칙이 `^영업이익`(시작만 보면 됨)이라서입니다.
- **현금흐름표(CF)의 '이자비용'은 손익계산서용 `interest_exp` 규칙에 안 잡힙니다.** 표 구분을 함께 걸어 두었기 때문입니다. 의도한 동작입니다.
- **'유동성 장기부채'는 공백을 지운 뒤 비교하므로 잡힙니다.** 처음에는 공백 때문에 이런 계정이 누락됐고, 공백 제거 비교로 고쳤습니다(인수인계 메모의 "수정 완료" 항목).
- **'유동 차입금'은 현재 규칙에 못 잡힙니다.** 파일럿 점검에서 확인된 변형인데 규칙에 아직 반영하지 않았습니다. 이것이 결정 **A15(차입 계정 매핑 확대)** 입니다. 규칙은 "첫 추정"이며 파일럿 결과를 보고 고치는 것이 원래 계획입니다.
- 규칙이 못 잡은 계정은 응답에 없는 것처럼 보이기 때문에, **"진짜 없음"과 "이름이 달라서 못 찾음"을 구분할 수 없게 됩니다.** 이 문제가 03번 노트북의 핵심 점검입니다.

---
# 3. `dart_api.py` — DART에 요청을 보내는 도구

이 파일은 세 가지 일을 합니다.
1. 인증키를 **환경변수**에서만 읽는다 (파일에 쓰거나 화면에 출력하지 않는다)
2. 요청을 보내고, 실패하면 몇 번 다시 시도한다
3. 응답의 **상태코드**(000, 013, 020 …)를 보고 "저장할지, 멈출지"를 정한다

> 이 노트북에서는 **진짜 요청을 보내지 않습니다.** 파이썬의 `mock`으로 "이런 응답이 왔다고 치자"를 만들어서 동작만 확인합니다.

### 3-1. 인증키 읽기 `api_key()`

In [6]:
print("지금 이 컴퓨터 창에 키가 설정되어 있나? (값은 출력하지 않음):", bool(os.environ.get("DART_API_KEY")))

# 키가 없는 상황을 흉내 내서 어떤 메시지가 나오는지만 본다 (진짜 환경변수는 건드리지 않음)
with mock.patch.dict(os.environ, {"DART_API_KEY": ""}):
    try:
        dart_api.api_key()
    except SystemExit as e:
        print("\n키가 없을 때의 동작 → 프로그램 중단, 메시지:")
        print(" ", e)

지금 이 컴퓨터 창에 키가 설정되어 있나? (값은 출력하지 않음): False

키가 없을 때의 동작 → 프로그램 중단, 메시지:
  환경변수 DART_API_KEY(40자리)가 설정되어 있지 않습니다. README의 '인증키 설정'을 따르세요.


### 해석
- 키는 **40자리**여야 하고, 아니면 바로 멈춥니다. 키를 코드에 직접 쓰지 않고 `os.environ`(컴퓨터 창이 열려 있는 동안만 기억하는 메모)에서만 읽는 이유는, 키가 파일에 남으면 깃(GitHub)에 올라가 **다른 사람이 내 키를 쓸 수 있기** 때문입니다.
- 이 세션의 셸에는 키가 설정되어 있지 않습니다. 수집을 돌릴 때는 **본인 PowerShell 창에서** `$env:DART_API_KEY = "..."`로 설정하고, **같은 창에서** Jupyter나 파이썬을 실행해야 합니다.

### 3-2. 요청 주소가 어떻게 만들어지나 `_open()`
요청은 **"주소 + ? + 이름=값&이름=값…"** 형태의 긴 주소입니다. 아래는 가짜 키로 만든 예입니다.

In [7]:
import urllib.parse

fake_key = "X" * 40
params = {"corp_code": "00126380", "bsns_year": "2022", "reprt_code": "11014", "fs_div": "CFS"}
query = urllib.parse.urlencode({**params, "crtfc_key": fake_key})
print(dart_api.BASE_URL + "fnlttSinglAcntAll.json?" + query)

https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json?corp_code=00126380&bsns_year=2022&reprt_code=11014&fs_div=CFS&crtfc_key=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


### 해석
- `corp_code`(어느 회사) · `bsns_year`(사업연도) · `reprt_code`(보고서 종류) · `fs_div`(연결/별도) 네 개가 "한 번의 질문"입니다. 위 예는 **삼성전자, 2022년, 3분기 보고서, 연결**입니다.
- 키(`crtfc_key`)가 주소 안에 들어갑니다. 그래서 코드는 오류가 났을 때 **주소를 메시지에 넣지 않습니다**(주소에 키가 들어 있어서). `_open()`의 `except` 부분을 보면 `HTTP 오류 코드`와 `경로`만 적습니다.
- `User-Agent` 헤더는 "나는 일반 브라우저다"라고 알리는 이름표입니다.

### 3-3. 상태코드를 보고 하는 일 `get_json()`
응답에는 항상 `status`(상태코드)가 들어 있습니다. 코드가 상태에 따라 다르게 행동하는 것을 **가짜 응답**으로 확인해 봅니다.

In [8]:
def fake_response(status, message="", extra=None):
    body = {"status": status, "message": message}
    if extra:
        body.update(extra)
    return json.dumps(body, ensure_ascii=False).encode("utf-8")

cases = [
    ("000", "정상",             {"list": [{"account_nm": "자산총계"}]}),
    ("013", "조회된 데이타가 없습니다.", None),
    ("020", "요청 제한을 초과하였습니다.", None),
    ("010", "등록되지 않은 키입니다.", None),
]

rows = []
for status, msg, extra in cases:
    with mock.patch.object(dart_api, "_with_retry", return_value=fake_response(status, msg, extra)):
        try:
            data = dart_api.get_json("fnlttSinglAcntAll.json", {})
            result = f"결과를 돌려줌 (status={data['status']})"
        except dart_api.LimitExceeded as e:
            result = "LimitExceeded 예외 → 오늘은 멈추고 내일 이어서"
        except dart_api.FatalApiStatus as e:
            result = "FatalApiStatus 예외 → 즉시 중단, 저장 안 함"
    rows.append((status, msg, result, "저장함" if status in dart_api.CACHEABLE_STATUSES else "저장 안 함"))
pd.DataFrame(rows, columns=["상태코드", "가짜 메시지", "get_json이 하는 일", "파일로 저장?"])

,상태코드,가짜 메시지,get_json이 하는 일,파일로 저장?
0,000,정상,결과를 돌려줌 (status=000),저장함
1,013,조회된 데이타가 없습니다.,결과를 돌려줌 (status=013),저장함
2,020,요청 제한을 초과하였습니다.,LimitExceeded 예외 → 오늘은 멈추고 내일 이어서,저장 안 함
3,010,등록되지 않은 키입니다.,"FatalApiStatus 예외 → 즉시 중단, 저장 안 함",저장 안 함


### 해석
| 상태 | 의미 | 왜 그렇게 처리하나 |
|---|---|---|
| **000** | 정상. 자료가 옴 | 저장. 다시 받지 않아도 되게 |
| **013** | 요청은 정상인데 **조회된 자료가 없음** | **저장**. "이 보고서는 없다"는 정보도 결과이고, 저장해 두어야 다음에 같은 질문을 또 하지 않음. 연결이 013이면 별도를 다시 물음 |
| **020** | 요청 제한 초과 | 오늘은 그만. 예외를 던져 반복문을 멈추고, 이미 받은 파일은 남아 있으니 **내일 같은 명령을 다시 실행하면 이어서** 받음 |
| **010·011·012·100·101·800·900·901** | 키/권한/서버 문제 | 계속 두드려도 소용없음. **응답을 저장하지 않고** 즉시 중단 |

- **"000·013만 저장"** 이 중요합니다. 오류 응답을 파일로 저장하면, 다음 실행이 그 파일을 "이미 받은 자료"로 믿고 건너뛰어서 **조용히 틀린 결과**가 됩니다. 실제로 한 번 이 오염 문제가 있었고 고쳤습니다.
- 위 상태코드의 뜻은 코드 주석(개발가이드의 오류 코드 표를 옮긴 것)에 근거합니다. 이번에 가이드 원문과 직접 대조하지는 않았습니다.

### 3-4. 실패하면 다시 시도 `_with_retry()`
네트워크가 잠깐 끊기는 일은 흔합니다. `_with_retry`는 실패하면 **2초, 4초 기다렸다가** 최대 3번 시도하고, 3번 다 실패하면 오류를 올립니다.

In [9]:
import time

calls = {"n": 0}
def flaky(path, params):
    calls["n"] += 1
    if calls["n"] < 3:
        raise dart_api.ApiError("네트워크 오류 (시험)")
    return b'{"status":"000"}'

with mock.patch.object(dart_api, "_open", side_effect=flaky), mock.patch.object(time, "sleep"):   # sleep은 시험이라 건너뜀
    out = dart_api._with_retry("company.json", {})
print("시도한 횟수:", calls["n"], "→ 3번째에 성공해서 결과를 받음:", out)

시도한 횟수: 3 → 3번째에 성공해서 결과를 받음: b'{"status":"000"}'


### 해석
두 번 실패해도 세 번째에 성공하면 그대로 진행합니다. 일시적 오류로 수집 전체가 멈추는 것을 막는 장치입니다. 단, **세 번 모두 실패하면 오류를 올리고**, 02번 노트북은 그 시점까지의 로그를 파일로 저장한 뒤 멈춥니다.

### 3-5. ZIP으로 오는 응답 `get_bytes()`
`corpCode.xml`(전체 법인 명단)만 JSON이 아니라 **ZIP 파일**로 옵니다. ZIP 파일은 맨 앞 두 글자가 `PK`입니다. 오류가 나면 ZIP이 아니라 글자 메시지가 오기 때문에, 맨 앞을 보고 구분합니다.

In [10]:
# 오류 메시지가 온 경우를 흉내 낸다
with mock.patch.object(dart_api, "_with_retry", return_value=b'{"status":"010","message":"not registered"}'):
    try:
        dart_api.get_bytes("corpCode.xml", {})
    except dart_api.ApiError as e:
        print("ZIP이 아닌 응답 → ApiError:", e)

# 정상 ZIP이 온 경우(맨 앞이 PK)
with mock.patch.object(dart_api, "_with_retry", return_value=b"PK\x03\x04...zip body..."):
    print("ZIP인 응답 → 그대로 돌려줌:", dart_api.get_bytes("corpCode.xml", {})[:2])

ZIP이 아닌 응답 → ApiError: ZIP이 아닌 응답이 왔습니다: {"status":"010","message":"not registered"}
ZIP인 응답 → 그대로 돌려줌: b'PK'


---
# 4. 정리: 어떤 노트북이 어떤 부품을 쓰나

| 노트북 | `config.py` | `dart_api.py` | 하는 일 |
|---|---|---|---|
| 01 명단·기업개황 | RAW, OUT, SLEEP, 업종 접두어, 이름 표시 규칙 | `get_bytes`, `get_json`, `CACHEABLE_STATUSES` | 법인 명단과 기업개황 받기 |
| 02 재무제표 수집 | YEARS, REPORTS, FS_ORDER, N_PILOT, SEED | `get_json` 외 예외 3종 | 기업×연도×보고서 재무제표 받기 |
| 03 추출 가능성 점검 | REPORTS, ACCOUNT_PATTERNS | 안 씀 (API 호출 없음) | 받은 자료에서 필요한 계정을 찾을 수 있는지 점검 |
| 04 정제표 예시 | REPORTS, ACCOUNT_PATTERNS | 안 씀 | 원자료를 기업×분기 표로 바꿔 보기 |

### 이 노트북에서 확인하지 못한 것
- 개발가이드 원문과 상태코드 표를 직접 대조하지 않았습니다(코드 주석 기준).
- `SLEEP=0.4`초가 서버 정책에 맞는 값인지 확인하지 않았습니다. 일일 요청 한도 값도 모릅니다.